# YouTube AdView Prediction

Machine Learning Regression Project

## 1. Import Libraries
We import only the required libraries to keep the project clean.

In [39]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import joblib


## 2. Load Dataset
We load the provided `train.csv` dataset and inspect the basic structure.

In [44]:

df = pd.read_csv("train.csv")
df.head()


,vidid,adview,views,likes,dislikes,comment,published,Duration,category
0,VID_18655,40,1031602,8523,363,1095,9/14/2016,457,F
1,VID_14135,2,1707,56,2,6,10/1/2016,570,D
2,VID_2187,1,2023,25,0,2,7/2/2016,136,C
3,VID_23096,6,620860,777,161,153,7/27/2016,262,H
4,VID_10175,1,666,1,0,0,6/29/2016,31,D


## 3. Data Cleaning
- Remove missing values
- Drop `vidid` as it does not help prediction

In [45]:

df = df.dropna()
df = df.drop(columns=["vidid"])
df.head()


,adview,views,likes,dislikes,comment,published,Duration,category
0,40,1031602,8523,363,1095,9/14/2016,457,F
1,2,1707,56,2,6,10/1/2016,570,D
2,1,2023,25,0,2,7/2/2016,136,C
3,6,620860,777,161,153,7/27/2016,262,H
4,1,666,1,0,0,6/29/2016,31,D


## 4. Feature Transformation
Convert duration into seconds and extract year from published date.

In [46]:

def duration_to_seconds(x):
    parts = str(x).split(":")
    parts = [int(p) for p in parts]
    if len(parts) == 3:
        return parts[0]*3600 + parts[1]*60 + parts[2]
    elif len(parts) == 2:
        return parts[0]*60 + parts[1]
    else:
        return parts[0]

df["Duration"] = df["Duration"].apply(duration_to_seconds)

df["published"] = pd.to_datetime(df["published"], errors="coerce")
df["published_year"] = df["published"].dt.year
df = df.drop(columns=["published"])
df.head()

,adview,views,likes,dislikes,comment,Duration,category,published_year
0,40,1031602,8523,363,1095,457,F,2016
1,2,1707,56,2,6,570,D,2016
2,1,2023,25,0,2,136,C,2016
3,6,620860,777,161,153,262,H,2016
4,1,666,1,0,0,31,D,2016


## 5. Encode Categorical Feature

In [47]:

le = LabelEncoder()
df["category"] = le.fit_transform(df["category"])
df.head()


,adview,views,likes,dislikes,comment,Duration,category,published_year
0,40,1031602,8523,363,1095,457,5,2016
1,2,1707,56,2,6,570,3,2016
2,1,2023,25,0,2,136,2,2016
3,6,620860,777,161,153,262,7,2016
4,1,666,1,0,0,31,3,2016


### 5.1 Convert Data Types

In [48]:
cols = ["views", "likes", "dislikes", "comment"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df.dropna(inplace=True)
df.head()

,adview,views,likes,dislikes,comment,Duration,category,published_year
0,40,1031602.0,8523.0,363.0,1095.0,457,5,2016
1,2,1707.0,56.0,2.0,6.0,570,3,2016
2,1,2023.0,25.0,0.0,2.0,136,2,2016
3,6,620860.0,777.0,161.0,153.0,262,7,2016
4,1,666.0,1.0,0.0,0.0,31,3,2016


## 6. Train-Test Split

In [49]:

X = df.drop(columns=["adview"])
y = df["adview"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 7. Linear Regression Model
This acts as a baseline model.

In [57]:

lr = LinearRegression()
lr.fit(X_train, y_train)

y_predict_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_predict_lr))
r2_lr = r2_score(y_test, y_predict_lr)
rmse_lr, r2_lr

(104264.29312691964, 0.0016687725640837092)

## 8. Random Forest Regressor
Random Forest handles non-linear relationships well and performs better on tabular data.

In [51]:

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

rmse_rf, r2_rf


(101180.30704537178, 0.05985371241849069)

## 9. Model Comparison

In [58]:

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "RMSE": [rmse_lr, rmse_rf],
    "R2 Score": [r2_lr, r2_rf]
})

results


,Model,RMSE,R2 Score
0,Linear Regression,104264.293127,0.001669
1,Random Forest,101180.307045,0.059854


## 10. Save Final Model
Random Forest is saved as the final model.

In [59]:

joblib.dump(rf, "youtube_adview_model.pkl")


['youtube_adview_model.pkl']

## Conclusion
Random Forest performed better than Linear Regression based on RMSE and R² score and was selected as the final model.